# 문장·단어·BERT·KLUE 복습

이 노트북은 `06_subword_tokenizer`의 두 수업 노트북을 바탕으로 SentencePiece, WordPiece, BERT 입력과 문맥 벡터를 총 30문제로 복습합니다.

- 각 문제 아래의 빈 코드 셀에 직접 풀이하세요.
- 설명 문제는 코드 셀에 주석으로 답해도 됩니다.
- 앞 문제에서 만든 변수를 뒤 문제에서 다시 사용하는 문제가 있으므로 위에서부터 실행하는 것을 권장합니다.
- `klue/bert-base`를 처음 불러올 때는 모델 파일 다운로드가 필요할 수 있습니다.
- 토큰 ID의 실제 숫자보다 **토큰과 ID의 위치 대응, mask 값, Tensor shape**에 집중하세요.

## 원본 노트북과 문제 연결

- `01_sentencepiece.ipynb`: 문제 1~14
- `02_BertWordPieceTokenizer.ipynb`: 문제 15~29
- 두 노트북 종합: 문제 30

문제 1~14는 직접 학습한 SentencePiece의 어휘와 ID를 읽고, 문제 15~29는 사전학습된 `klue/bert-base`의 토크나이저와 Encoder를 사용합니다.

In [ ]:
from pathlib import Path

import pandas as pd
import sentencepiece as spm
import torch

# 노트북을 복습자료 폴더 또는 프로젝트 루트에서 열어도 원본 경로를 찾는다.
source_candidates = [
    Path("../../06_subword_tokenizer"),
    Path("06_subword_tokenizer"),
]
SOURCE_DIR = next(path for path in source_candidates if path.exists())
ARTIFACT_DIR = SOURCE_DIR / "sentencepiece_artifacts"

TRAIN_PATH = ARTIFACT_DIR / "ratings_train.txt"
SP_MODEL_PATH = ARTIFACT_DIR / "nsmc.model"
SP_VOCAB_PATH = ARTIFACT_DIR / "nsmc.vocab"
MODEL_NAME = "klue/bert-base"

print("원본 폴더:", SOURCE_DIR.resolve())
print("NSMC 파일:", TRAIN_PATH.exists())
print("SentencePiece 모델:", SP_MODEL_PATH.exists())

# 1. SentencePiece와 서브워드

## 문제 1. 단어 OOV와 서브워드 비교

단어 단위 어휘가 `재미있다`만 알고 있고 새 입력으로 `재미있지만`이 들어왔다고 가정합니다.

- OOV의 풀네임과 뜻을 작성하세요.
- 단어 단위 토큰화에서 어떤 정보가 사라질 수 있는지 설명하세요.
- `재미있지만`을 재사용 가능한 두 개 이상의 서브워드로 나누어 보세요.
- 서브워드가 OOV를 **줄일 수는 있지만 완전히 없애지는 못하는** 이유를 한 문장으로 작성하세요.

## 문제 2. Unigram과 BPE 구분

SentencePiece에서 사용할 수 있는 두 학습 방식을 비교하세요.

| 비교 기준 | Unigram | BPE |
| --- | --- | --- |
| 시작 상태 |  |  |
| 반복 동작 |  |  |
| 최종 분할을 결정하는 기준 |  |  |

추가로 다음 설명이 어느 방식인지 답하세요.

1. 자주 붙어 등장하는 이웃 조각 쌍을 반복해서 합친다.
2. 많은 후보 조각에서 시작해 도움이 적은 조각을 제거한다.

## 문제 3. NSMC에서 토크나이저 학습 코퍼스 만들기

`TRAIN_PATH`의 TSV 파일을 읽어 SentencePiece 학습용 문장 목록을 준비하세요.

조건:

- `pd.read_csv(..., sep="\t")`로 `raw_data`를 만드세요.
- `document`가 결측인 행을 제거하고 인덱스를 초기화한 결과를 `clean_data`에 저장하세요.
- 정제 전후 shape, 열 이름, 남은 결측치 수를 출력하세요.
- SentencePiece 학습에는 `document`만 사용하고 `id`, `label`은 제외하세요.
- 감성 레이블을 사용하지 않아도 되는 이유를 주석으로 설명하세요.

## 문제 4. SentencePiece 학습 설정 읽기

다음 요구사항을 만족하는 `SentencePieceTrainer.train()` 호출 코드를 작성하세요. 전체 NSMC 학습은 시간이 걸릴 수 있으므로 설정을 작성한 뒤 실행 여부는 선택해도 됩니다.

- 입력: 한 줄에 리뷰 하나가 저장된 `nsmc.txt`
- 모델 prefix: `nsmc`
- 방식: Unigram
- 전체 어휘 수: 30,000
- `[PAD]`, `[OOV]`, `[BOS]`, `[EOS]`의 ID: 각각 0, 1, 2, 3
- 각 특수 토큰의 문자열도 직접 지정

`vocab_size`를 너무 작게 또는 너무 크게 정했을 때 생길 수 있는 문제도 각각 한 문장으로 작성하세요.

## 문제 5. `.model`과 `.vocab`의 역할 확인

이미 학습된 `SP_MODEL_PATH`와 `SP_VOCAB_PATH`를 사용하세요.

- `SentencePieceProcessor(model_file=...)`로 `sp_model`을 불러오세요.
- 실제 어휘 수와 두 파일의 존재 여부를 출력하세요.
- `.model`과 `.vocab`이 각각 어떤 목적으로 필요한지 주석으로 작성하세요.
- 실제 인코딩에 반드시 필요한 파일이 무엇인지 답하세요.

## 문제 6. 특수 토큰 ID와 역할 검증

문제 5의 `sp_model`에서 PAD, OOV, BOS, EOS ID를 조회하세요.

- `pad_id()`, `unk_id()`, `bos_id()`, `eos_id()`를 이용해 `special_ids` 딕셔너리를 만드세요.
- 예상값 `{PAD: 0, OOV: 1, BOS: 2, EOS: 3}`과 같은지 확인하세요.
- 각 ID를 `id_to_piece()`로 다시 문자열 조각으로 바꾸세요.
- PAD와 OOV가 같은 ID이면 안 되는 이유를 attention mask와 연결해 설명하세요.
- 이 번호가 모든 SentencePiece 모델에서 항상 같다고 말할 수 있는지도 답하세요.

## 문제 7. 문자열 조각과 토큰 ID의 위치 대응

문장 `영화가 재미있지만 결말은 아쉽다`를 두 방식으로 인코딩하세요.

- `out_type=str` 결과를 `pieces`, `out_type=int` 결과를 `token_ids`에 저장하세요.
- 두 목록의 길이가 같은지 확인하세요.
- `zip()`과 `enumerate()`를 사용해 위치, 조각, ID를 한 줄씩 출력하세요.
- ID 숫자의 크기가 토큰의 빈도, 중요도 또는 의미적 크기를 뜻하는지 설명하세요.

## 문제 8. SentencePiece의 `▁` 공백 경계 읽기

문장 `오늘 영화 정말 재미있다`를 조각으로 인코딩하고 `▁`가 포함된 조각만 출력하세요.

- `▁`가 원문의 어떤 정보를 보존하는지 설명하세요.
- `▁영화`와 `영화`를 읽는 관점의 차이를 작성하세요.
- WordPiece의 `##`와 SentencePiece의 `▁`가 모두 경계 표시이지만 방향은 어떻게 다른지 예상해 보세요.

## 문제 9. OOV가 발생한 정확한 위치 찾기

문장 `영화가 재미있지만 🧪 표현은 낯설다`를 조각과 ID로 인코딩하세요.

- `sp_model.unk_id()`를 `oov_id`에 저장하세요.
- OOV ID가 나타난 0부터 시작하는 위치를 `oov_positions`에 모으세요.
- 각 OOV 위치의 piece와 ID를 함께 출력하세요.
- piece 출력에 원래 문자가 보이더라도 ID가 `oov_id`라면 모델이 그 문자의 전용 임베딩을 조회하는 것인지 설명하세요.

## 문제 10. Encode와 decode 왕복 변환

다음 두 문장을 각각 `원문 → ID 목록 → 복원문`으로 변환하세요.

```python
decode_examples = [
    "아 더빙 정말 재미있다",
    "영화가 재미있지만 🧪 표현은 낯설다",
]
```

- 각 문장의 원문, ID, 복원문, 완전 일치 여부를 출력하세요.
- OOV 또는 정규화가 있을 때 원문과 완전히 같지 않을 수 있는 이유를 설명하세요.
- `decode()`가 신경망 모델의 예측을 되돌리는 과정인지, 토큰화의 역변환인지 구분하세요.

## 문제 11. 오른쪽 패딩과 잘림 구현

아래 세 문장을 `max_length=12`로 맞추세요.

```python
sentences = [
    "재미있다",
    "영화가 정말 재미있다",
    "영화가 재미있지만 중간 이야기가 너무 길고 마지막 결말은 아쉬웠다",
]
```

- 먼저 각 문장을 ID 목록으로 인코딩하세요.
- 12개를 넘는 ID는 뒤를 자르고, 짧은 목록에는 오른쪽에 PAD ID를 추가하세요.
- 결과를 `padded_ids`에 저장하고 모든 행의 길이가 12인지 확인하세요.
- 각 문장의 원본 토큰 수와 잘림 여부를 출력하세요.

## 문제 12. Attention mask 직접 만들기

문제 11의 `padded_ids`로 `attention_mask`를 만드세요.

- PAD ID 위치만 0, 나머지는 모두 1로 지정하세요.
- OOV ID가 있다면 mask가 0인지 1인지 확인하고 이유를 설명하세요.
- 각 행에서 `sum(mask)`가 무엇을 뜻하는지 작성하세요.
- `padded_ids`와 `attention_mask`의 shape가 같아야 하는 이유를 설명하세요.

## 문제 13. 패딩 결과의 불변 조건 검사

문제 11~12의 결과가 올바른지 `assert`로 검사하세요.

- 배치 크기는 3이다.
- 모든 ID 행과 mask 행의 길이는 12이다.
- PAD 위치의 mask는 항상 0이다.
- PAD가 아닌 위치의 mask는 항상 1이다.
- 각 mask에는 0과 1 이외의 값이 없다.

검사를 모두 통과하면 `패딩과 mask 검사 통과`를 출력하세요.

## 문제 14. SentencePiece 코드 오류 찾기

다음 코드에는 메서드 호출과 mask 작성에 관한 오류가 있습니다. 올바르게 수정하고 이유를 설명하세요.

```python
pad_id = sp_model.pad_id
oov_id = sp_model.unk_id()
row = [15, oov_id, 27, pad_id, pad_id]
mask = [0 if token_id in (pad_id, oov_id) else 1 for token_id in row]
```

조건:

- `pad_id`에는 정수 ID가 저장되어야 합니다.
- 실제 입력인 OOV 위치는 1, 빈자리 PAD 위치만 0이어야 합니다.
- 수정 후 `row`와 `mask`를 출력하세요.

# 2. WordPiece와 KLUE BERT

## 문제 15. 단어 단위 UNK와 WordPiece 비교

다음 작은 단어 어휘와 문장을 사용하세요.

```python
word_vocab = {"영화가", "재미", "있다"}
text = "영화가 재미있다"
```

- 공백으로 분리한 단어가 `word_vocab`에 없으면 `[UNK]`로 바꾸어 `word_level_tokens`를 만드세요.
- `AutoTokenizer.from_pretrained(MODEL_NAME)`로 `tokenizer`를 불러오세요.
- `tokenizer.tokenize(text)` 결과를 `wordpiece_tokens`에 저장하세요.
- 두 결과를 비교하여 WordPiece가 단어 전체의 OOV를 어떻게 줄이는지 설명하세요.

## 문제 16. WordPiece의 `##` 경계 읽기

문장 `영화가 재미있다`의 WordPiece 조각을 출력하세요.

- `##`로 시작하는 조각만 따로 모으세요.
- `영화`, `##가`가 두 개의 독립된 단어를 뜻하는지 설명하세요.
- `##`가 원문에 실제로 존재하는 문자 표시인지 답하세요.
- SentencePiece의 `▁`와 WordPiece의 `##`를 단어의 시작과 이어짐 관점에서 비교하세요.

## 문제 17. WordPiece 조각과 ID 왕복 조회

문제 16의 조각을 KLUE BERT 어휘 ID로 바꾸세요.

- `convert_tokens_to_ids()` 결과를 `token_ids`에 저장하세요.
- 위치, 조각, ID를 한 줄씩 출력하세요.
- `convert_ids_to_tokens()`로 다시 조각을 조회하세요.
- 처음 조각과 복원 조각이 같은지 확인하세요.
- 토큰 ID가 임베딩 행렬과 어떤 관계인지 한 문장으로 작성하세요.

## 문제 18. BERT 특수 토큰 역할 구분

`[PAD]`, `[UNK]`, `[CLS]`, `[SEP]`, `[MASK]`의 ID를 KLUE BERT 어휘에서 조회해 딕셔너리로 만드세요.

각 토큰의 역할도 한 문장씩 작성하세요.

- 문장 분류에서 대표 위치로 사용할 수 있는 토큰
- 문장 끝 또는 문장 쌍의 경계를 표시하는 토큰
- 길이를 맞추는 빈자리 토큰
- 표현할 수 없는 입력을 대신하는 토큰
- BERT 사전학습에서 원래 토큰을 맞히게 하는 토큰

특수 토큰의 ID가 다른 checkpoint에서도 반드시 같은지 답하세요.

## 문제 19. Attention mask와 special tokens mask 비교

다음 토큰의 두 mask를 직접 완성하세요.

```text
tokens:              [CLS] 영화 ##가 [SEP] [PAD] [PAD]
attention_mask:         ?    ?    ?     ?     ?     ?
special_tokens_mask:    ?    ?    ?     ?     ?     ?
```

- `[CLS]`와 `[SEP]`의 attention mask가 1인 이유를 설명하세요.
- `[PAD]`의 special tokens mask와 attention mask 값이 서로 다른 이유를 설명하세요.
- `[MASK]` 토큰과 `attention_mask` 배열이 전혀 다른 개념인 이유를 작성하세요.
- 현재 BERT 순전파에 직접 전달할 mask가 어느 것인지 답하세요.

## 문제 20. 한 문장을 BERT 입력으로 한 번에 변환

문장 `영화가 재미있다 🧪`를 `tokenizer()`로 변환하세요.

조건:

- `padding="max_length"`, `truncation=True`, `max_length=12`를 사용하세요.
- `return_special_tokens_mask=True`를 지정하세요.
- 반환된 key와 각 목록을 출력하세요.
- `input_ids`를 다시 토큰 문자열로 바꾸어 `[CLS]`, `[SEP]`, `[PAD]`, `[UNK]` 위치를 확인하세요.
- attention mask에서 0인 위치의 토큰이 모두 `[PAD]`인지 검사하세요.

## 문제 21. 여러 문장을 PyTorch 배치로 만들기

다음 두 문장을 길이 12의 Tensor 배치로 만드세요.

```python
batch_texts = [
    "영화가 재미있다",
    "영화가 재미있지만 결말은 그래도 아쉬웠다",
]
```

- 문제 20의 옵션에 `return_tensors="pt"`를 추가해 `batch_encoding`을 만드세요.
- 각 key의 Tensor shape와 dtype을 출력하세요.
- `input_ids`, `attention_mask`, `token_type_ids`가 모두 `(2, 12)`인지 확인하세요.
- 문장별 토큰 목록, attention mask, 모델이 읽을 위치 수를 출력하세요.

## 문제 22. `token_type_ids`로 문장 쌍 구분하기

다음 문장 쌍 하나를 BERT 입력으로 만드세요.

```python
sentence_a = "영화가 재미있다"
sentence_b = "결말은 아쉽다"
```

- `tokenizer(sentence_a, sentence_b, ...)` 형식으로 길이 16의 입력을 만드세요.
- 토큰과 `token_type_ids`를 같은 위치끼리 출력하세요.
- 첫 문장 구간과 둘째 문장 구간에 각각 어떤 값이 사용되는지 확인하세요.
- 단일 문장 입력에서 `token_type_ids`가 모두 0이었던 이유를 설명하세요.

## 문제 23. 패딩과 잘림 결과 읽기

문제 21의 두 문장을 이번에는 `max_length=8`로 토큰화한 결과와 비교하세요.

- 길이 12와 길이 8에서 각 문장의 실제 토큰 수를 attention mask 합으로 구하세요.
- 길이 8 결과의 토큰 목록에서 `[CLS]`와 `[SEP]`가 유지되는지 확인하세요.
- 긴 문장에서 어떤 원문 조각이 사라졌는지 토큰 목록을 비교해 찾으세요.
- `max_length`가 너무 작을 때 분류 모델에 생길 수 있는 문제를 설명하세요.

## 문제 24. 토크나이저와 모델 checkpoint 맞추기

`AutoTokenizer.from_pretrained(MODEL_NAME)`와 `AutoModel.from_pretrained(MODEL_NAME)`의 이름을 같게 유지해야 하는 이유를 설명하세요.

다음 흐름을 완성해 작성하세요.

```text
문자열 → (        ) → input ID → (        )의 해당 행 → 토큰 임베딩
```

서로 다른 checkpoint의 토크나이저와 모델을 섞으면 ID 숫자가 같더라도 왜 다른 조각의 임베딩을 조회할 수 있는지 설명하세요.

## 문제 25. `train()`, `eval()`, `inference_mode()` 구분

`AutoModel.from_pretrained(MODEL_NAME)`로 `bert_model`을 불러온 뒤 평가 모드로 바꾸세요.

- `eval`의 풀네임을 작성하세요.
- `bert_model.eval()`이 바꾸는 모델 동작을 Dropout 관점에서 설명하세요.
- `eval()`만 호출하면 gradient 계산도 자동으로 꺼지는지 답하세요.
- 추론 순전파를 `with torch.inference_mode():` 안에서 실행하는 이유를 설명하세요.
- 다시 미세조정 학습을 시작할 때 호출해야 하는 메서드도 작성하세요.

## 문제 26. BERT 입력과 출력 shape 추적

문제 21의 `batch_encoding`에서 BERT가 받는 세 Tensor만 선택해 `model_inputs`를 만드세요.

- 선택할 key는 `input_ids`, `attention_mask`, `token_type_ids`입니다.
- 설명용 `special_tokens_mask`는 제외하세요.
- `torch.inference_mode()`에서 `bert_model(**model_inputs)`를 실행하세요.
- `last_hidden_state`의 shape를 출력하세요.
- 입력 `(2, 12)`가 출력 `(2, 12, 768)`로 바뀔 때 각 축이 무엇을 뜻하는지 설명하세요.
- 768이 입력 문장으로부터 계산된 가변적인 숫자인지 모델 설정의 `hidden_size`인지 확인하세요.

## 문제 27. `[CLS]` 문장 벡터 선택하기

문제 26의 `last_hidden_state`에서 모든 문장의 0번 토큰 위치를 선택해 `cls_vectors`를 만드세요.

- 슬라이싱은 `[:, 0, :]`를 사용하세요.
- 결과 shape가 `(2, 768)`인지 확인하세요.
- 입력 전의 `[CLS]` 토큰 ID와 BERT 통과 후의 `[CLS]` 출력 벡터가 어떻게 다른지 설명하세요.
- `cls_vectors` 자체가 긍정·부정 두 클래스의 점수인지 답하세요.
- 감성 분류 점수를 만들려면 뒤에 어떤 층과 학습이 더 필요한지 작성하세요.

## 문제 28. Self-Attention과 문맥적 토큰 벡터

다음 두 문장에서 같은 표면형 `은행`이 서로 다른 문맥을 갖습니다.

```text
A. 은행에서 돈을 찾았다.
B. 가을에 은행 열매를 주웠다.
```

- BERT에서 각 `은행` 토큰의 출력 벡터가 서로 달라질 수 있는 이유를 Self-Attention으로 설명하세요.
- Self-Attention이 다른 토큰을 모두 같은 비율로 섞는 연산인지 답하세요.
- 각 토큰이 문장 속 다른 토큰을 참고하여 자신의 벡터를 갱신한다는 설명을 쉬운 말로 다시 작성하세요.
- Word2Vec 같은 정적 임베딩과 BERT 문맥 벡터의 차이를 한 문장으로 비교하세요.

## 문제 29. Attention mask를 이용한 평균 풀링

PAD 위치의 출력까지 단순 평균하면 문장 벡터가 왜곡될 수 있습니다. 문제 26의 결과로 PAD를 제외한 평균 벡터를 만드세요.

조건:

- `attention_mask`를 `(2, 12, 1)`로 확장하세요.
- `last_hidden_state`와 곱해 PAD 위치를 0으로 만드세요.
- 토큰 축으로 합한 뒤 문장별 유효 토큰 수로 나누세요.
- 결과 `mean_vectors`의 shape가 `(2, 768)`인지 확인하세요.
- `[CLS]` 선택과 mask 평균 풀링이 문장을 대표하는 서로 다른 방법임을 설명하세요.

## 문제 30. 최종 종합: 문자열에서 분류 점수까지

다음 전체 흐름을 순서대로 설명하고, 각 단계의 대표 객체 또는 Tensor shape를 작성하세요.

```text
문자열
→ 서브워드 조각
→ 특수 토큰을 포함한 input_ids
→ attention_mask·token_type_ids와 함께 BERT 입력
→ last_hidden_state
→ CLS 문장 벡터
→ 분류 로짓
```

실습 코드도 작성하세요.

- 새 문장 두 개를 길이 16으로 토큰화하세요.
- BERT Encoder를 평가 모드에서 실행하세요.
- `(2, 768)` CLS 벡터를 얻으세요.
- `torch.nn.Linear(768, 2)` 분류 헤드에 전달하여 `(2, 2)` 로짓을 만드세요.
- 무작위로 초기화한 분류 헤드의 로짓을 실제 감성 예측으로 믿으면 안 되는 이유를 설명하세요.
- SentencePiece를 새로 학습하는 방식과 KLUE BERT의 기존 WordPiece를 사용하는 방식의 가장 큰 차이를 정리하세요.